# Robustness check — how many replications do we need?

**Goal:** Determine empirically whether 100 replications is necessary, or whether
50 (or even 20) are sufficient for the figures in this paper.

**Approach:**
1. **Part I — Variability map:** Load notebook 01 CSVs (already computed) and examine
   where in the (b, θ) space the variability is highest. This tells us the *worst case*.
2. **Part II — Rep-count analysis:** Run a large batch (N_MAX reps) at the worst-case
   point and several representative points. Subsample to n = 5, 10, 20, 50, 100, 200
   and show how the mean estimate and its standard error evolve.
3. **Part III — Visual quality:** Show what the full (b, θ) heatmap looks like
   when subsampled to different rep counts. Decide by eye.

Network: BA z=4, L=1, Fermi (results from notebook 01 + a targeted new run).

In [ ]:
import sys
sys.path.insert(0, '..')   # import model from parent directory

import os
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import model

# ── Parameters (must match notebook 01) ─────────────────────────────────────
N        = 1000
L        = 1
lam      = 0.5
K_fermi  = 0.1
b_vals   = np.round(np.linspace(1.0, 2.0, 11), 2)
th_vals  = np.round(np.linspace(0.0, 1.0, 11), 2)
NET_SEED = 0
N_JOBS   = -1

# ── New simulation parameters ────────────────────────────────────────────────
N_MAX    = 300    # total reps to run for the targeted analysis
N_BOOT   = 1000  # bootstrap resamples per n_rep value
N_VALUES = [5, 10, 20, 50, 100, 200, 300]   # subsample sizes to evaluate

os.makedirs('data',    exist_ok=True)
os.makedirs('figures', exist_ok=True)

db, dt = b_vals[1]-b_vals[0], th_vals[1]-th_vals[0]
EXTENT  = [b_vals[0]-db/2, b_vals[-1]+db/2, th_vals[0]-dt/2, th_vals[-1]+dt/2]

def rho_matrix(df, col='rho_mean'):
    return (
        df.pivot(index='theta', columns='b', values=col)
          .sort_index(ascending=True).values
    )

---
## Part I — Variability map from notebook 01
*Requires `../data/01-BA_z4-fermi.csv` to exist (run notebook 01 first).*

In [ ]:
# ── Load notebook 01 results ─────────────────────────────────────────────────
df01 = pd.read_csv('../data/01-BA_z4-fermi.csv')
print(df01.describe())
print(f"\nMax std:  {df01['rho_std'].max():.4f}  at b={df01.loc[df01['rho_std'].idxmax(),'b']}, "
      f"θ={df01.loc[df01['rho_std'].idxmax(),'theta']}")
print(f"Mean std: {df01['rho_std'].mean():.4f}")
print(f"Max SEM:  {(df01['rho_std']/np.sqrt(100)).max():.4f}   (SEM = std / sqrt(100))")

In [ ]:
# ── Fig 1a: heatmap of rho_std across (b, θ) ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)

for ax, col, title, cmap in [
    (axes[0], 'rho_mean', '$\\langle\\rho\\rangle$',   'Blues'),
    (axes[1], 'rho_std',  'std$(\\rho)$',              'Oranges'),
    (axes[2], 'rho_std',  'SEM = std$/\\sqrt{100}$',  'Reds'),
]:
    mat = rho_matrix(df01, col)
    if col == 'rho_std' and title.startswith('SEM'):
        mat = mat / np.sqrt(100)
    im = ax.imshow(mat, origin='lower', aspect='auto', cmap=cmap, extent=EXTENT)
    ax.set_xlabel('$b$'); ax.set_ylabel('$\\theta$')
    ax.set_title(title)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(0.5))
    ax.yaxis.set_major_locator(mticker.MultipleLocator(0.5))
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)

fig.suptitle('BA z=4, Fermi, L=1 — variability across (b, θ)  [n=100 reps]')
fig.savefig('figures/01-variability-map.png', dpi=150, bbox_inches='tight')
plt.show()

# Identify the worst-case point
idx_worst = df01['rho_std'].idxmax()
B_WORST  = float(df01.loc[idx_worst, 'b'])
TH_WORST = float(df01.loc[idx_worst, 'theta'])
print(f'Worst-case point: b={B_WORST}, θ={TH_WORST},  std={df01.loc[idx_worst,"rho_std"]:.4f}')

---
## Part II — Rep-count analysis at representative points
*Runs N_MAX=300 replications at a few (b, θ) points; subsamples to study convergence.*

In [ ]:
# ── Build network and compile JIT ────────────────────────────────────────────
G  = model.build_network('BA', N, 4, seed=NET_SEED)
gp, gd = model.game_csr(G)
sp, sd = model.shells_csr(G, L)
al     = model.geometric_kernel(L, lam)
model.warm_up()
print(f'Network: N={G.number_of_nodes()}, <k>={2*G.number_of_edges()/G.number_of_nodes():.2f}')
print('JIT compiled.')

In [ ]:
# ── Define test points ───────────────────────────────────────────────────────
# worst-case + two representative points (cooperative / defective)
TEST_POINTS = [
    (B_WORST, TH_WORST, 'worst case'),
    (1.3, 0.3, 'cooperative regime'),
    (1.8, 0.7, 'defective regime'),
]
print('Test points:', TEST_POINTS)

In [ ]:
# ── Run N_MAX reps at each test point (cached) ───────────────────────────────
rhos_by_point = {}

for b, theta, label in TEST_POINTS:
    tag  = f"b{b:.1f}_th{theta:.1f}".replace('.','p')
    path = f"data/reps_{tag}.npy"
    if os.path.exists(path):
        print(f"  loading {path}")
        rhos_by_point[(b, theta)] = np.load(path)
    else:
        print(f"  running {label} (b={b}, θ={theta}, {N_MAX} reps) ...", flush=True)
        rhos = model.run_replications(
            gp, gd, sp, sd, al,
            b=b, theta=theta, K=K_fermi,
            n_rep=N_MAX, n_jobs=N_JOBS, base_seed=1000,
        )
        np.save(path, rhos)
        rhos_by_point[(b, theta)] = rhos
        print(f"  saved {path}  mean={rhos.mean():.3f}  std={rhos.std():.3f}")

print('Done.')

In [ ]:
# ── Bootstrap: SEM and std as function of n_rep ──────────────────────────────
rng = np.random.default_rng(42)

boot_stats = {}   # (b, theta, n) → {'sem': float, 'std': float}

for b, theta, label in TEST_POINTS:
    rhos = rhos_by_point[(b, theta)]
    for n in N_VALUES:
        if n > len(rhos):
            continue
        # Bootstrap: resample n reps N_BOOT times, compute mean each time
        boot_means = np.array([
            rhos[rng.choice(len(rhos), n, replace=False)].mean()
            for _ in range(N_BOOT)
        ])
        boot_stats[(b, theta, n)] = {
            'sem':       boot_means.std(),          # std of the mean estimate
            'mean_std':  rhos[:n].std(ddof=1),      # within-sample std
            'mean_rho':  boot_means.mean(),
        }

print('Bootstrap done.')

In [ ]:
# ── Fig 2: SEM vs n_rep for each test point ──────────────────────────────────
fig2, axes2 = plt.subplots(1, 2, figsize=(9, 4), constrained_layout=True)

colors = {'worst case': 'C3', 'cooperative regime': 'C0', 'defective regime': 'C2'}

for b, theta, label in TEST_POINTS:
    ns   = [n for n in N_VALUES if (b, theta, n) in boot_stats]
    sems = [boot_stats[(b, theta, n)]['sem'] for n in ns]
    stds = [boot_stats[(b, theta, n)]['mean_std'] for n in ns]
    c    = colors[label]

    axes2[0].plot(ns, sems, 'o-', color=c, label=f'{label} (b={b}, θ={theta})', lw=1.8)
    axes2[1].plot(ns, stds, 'o-', color=c, label=f'{label}', lw=1.8)

# Add theoretical 1/sqrt(n) reference
ns_ref = np.array(N_VALUES)
worst_std = rhos_by_point[(TEST_POINTS[0][0], TEST_POINTS[0][1])].std()
axes2[0].plot(ns_ref, worst_std / np.sqrt(ns_ref), 'k--', lw=1, alpha=0.5,
              label='$\\sigma/\\sqrt{n}$ reference')

for ax, ylabel, title in [
    (axes2[0], 'SEM  (std of $\\hat{\\rho}$)',   'Standard error of the mean vs n_rep'),
    (axes2[1], 'std$(\\rho)$ within sample',       'Within-sample std vs n_rep'),
]:
    ax.set_xlabel('Number of replications')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    # Mark key n values
    for n_mark in [20, 50, 100]:
        ax.axvline(n_mark, color='grey', lw=0.8, ls=':')
        ax.text(n_mark*1.05, ax.get_ylim()[1]*0.8, str(n_mark),
                fontsize=7, color='grey', va='top')

fig2.suptitle('Bootstrap SEM vs number of replications — BA z=4, L=1, Fermi')
fig2.savefig('figures/02-sem-vs-nrep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/02-sem-vs-nrep.png')

In [ ]:
# ── Summary table ────────────────────────────────────────────────────────────
print(f"{'n_rep':>8} | {'SEM worst':>12} | {'SEM coop':>12} | {'SEM defect':>12}")
print('-' * 52)
for n in N_VALUES:
    row = []
    for b, theta, label in TEST_POINTS:
        if (b, theta, n) in boot_stats:
            row.append(f"{boot_stats[(b,theta,n)]['sem']:.4f}")
        else:
            row.append('  n/a ')
    print(f"{n:>8} | {row[0]:>12} | {row[1]:>12} | {row[2]:>12}")

---
## Part III — Visual quality of heatmaps at different n_rep
*Subsamples the notebook 01 data (100 reps) to show what heatmaps look like with fewer reps.*

Note: we only have the mean and std from notebook 01, not the individual replication values.
Here we simulate the effect of n_rep by adding noise scaled to match the observed std:
a reduced-n heatmap has SEM ≈ std / sqrt(n), which we add as noise to the mean.

In [ ]:
# ── Simulated heatmaps at different effective n_rep ───────────────────────────
# We add zero-mean Gaussian noise with std = rho_std / sqrt(n_rep / 100)
# to simulate what the heatmap looks like with fewer replications.
N_SHOW   = [10, 20, 50, 100]   # rep counts to display
rng_vis  = np.random.default_rng(0)

mean_mat = rho_matrix(df01, 'rho_mean')
std_mat  = rho_matrix(df01, 'rho_std')

fig3, axes3 = plt.subplots(1, len(N_SHOW), figsize=(12, 3.5), constrained_layout=True)

for ax, n in zip(axes3, N_SHOW):
    noise    = rng_vis.normal(0, std_mat / np.sqrt(n / 100), mean_mat.shape)
    mat_noisy = np.clip(mean_mat + noise, 0, 1)
    im = ax.imshow(
        mat_noisy, origin='lower', aspect='auto',
        vmin=0, vmax=1, cmap='Blues', extent=EXTENT,
    )
    ax.set_title(f'n = {n} reps')
    ax.set_xlabel('$b$'); ax.set_ylabel('$\\theta$')
    ax.xaxis.set_major_locator(mticker.MultipleLocator(0.5))
    ax.yaxis.set_major_locator(mticker.MultipleLocator(0.5))
    fig3.colorbar(im, ax=ax, fraction=0.046, pad=0.03).set_ticks([0, 0.5, 1])

fig3.suptitle('Visual quality of ⟨ρ⟩ heatmap as function of n_rep  (BA z=4, Fermi, L=1)')
fig3.savefig('figures/03-heatmap-nrep-comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/03-heatmap-nrep-comparison.png')

---
## Conclusions

*(Fill in after running)*

- Where is the variability highest in the (b, θ) space? (Near the cooperation-defection boundary?)
- What is the SEM at n=50 vs n=100? Is the difference meaningful for the paper figures?
- At what n_rep do the heatmaps look visually stable?
- **Decision:** use n_rep = __ for notebooks 02, 03, 04.